# 04 · 转动一盒彩色点，看看画面怎样变化

想象空间里飘着一盒彩色图钉。你绕着它们转动，相同的点会落到屏幕上不同的位置；固定一个视角，又可以调整颜色，让画面向目标图靠近。

这篇保留旧交互实验，做两件事：**用鼠标观察点云，用像素损失调整点的颜色。** 沿着代码读，还能看清“声明了参数”和“参数真正参与绘图”之间的联系。

## 打开方式

使用本主题环境和支持 OpenCV 窗口的本机内核。`RUN_INTERACTIVE=False` 为阅读模式；改为 `True` 并从头运行后，进入交互窗口。

- 左键拖动：旋转视角。
- 滚轮：缩放。
- 空格：切换颜色拟合与查看模式。
- q：退出窗口。

默认使用 2000 个点和 320×240 目标图。下面先读投影，再读着色，最后进入窗口。

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image

RUN_INTERACTIVE = False
NUM_POINTS = 2000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if RUN_INTERACTIVE:
    import cv2
print("GUI enabled:", RUN_INTERACTIVE, "Device:", device)

## 1. 从空间里的点，算到屏幕上的位置

四元数 `(w,x,y,z)` 描述方向，归一化后转换成旋转矩阵 R；尺度 S 与它组合成 `L=RS`，协方差由 `Σ=LLᵀ` 得到。

`project_gaussians` 再把中心变到相机坐标，用视场角计算焦距，返回屏幕中心、二维协方差和深度。

<details>
<summary>这段旧投影的两处调试处理</summary>

代码把深度筛选 mask 全设为 True，并用 `abs(z)` 参与投影，因此相机后方的点也可能落入画面。`projmatrix` 参数保留在接口中，实际焦距由视场角计算。

这一版用于观察旧点云演示。需要一致的 OpenCV 正深度几何时，使用[第一篇的相机实现](01_相机与高斯投影.ipynb)。

</details>

In [ ]:
def build_rotation(r):
    """
    构建旋转矩阵 (Batch)
    r: (N, 4) 四元数 [w, x, y, z]
    这里为了简单，我们假设输入是四元数，因为四元数在优化时更稳定
    """
    # 归一化四元数
    norm = torch.sqrt(r[:,0]*r[:,0] + r[:,1]*r[:,1] + r[:,2]*r[:,2] + r[:,3]*r[:,3])
    q = r / norm[:, None]

    R = torch.zeros((q.shape[0], 3, 3), device=q.device)

    r = q[:, 0]
    x = q[:, 1]
    y = q[:, 2]
    z = q[:, 3]

    R[:, 0, 0] = 1 - 2 * (y*y + z*z)
    R[:, 0, 1] = 2 * (x*y - r*z)
    R[:, 0, 2] = 2 * (x*z + r*y)
    R[:, 1, 0] = 2 * (x*y + r*z)
    R[:, 1, 1] = 1 - 2 * (x*x + z*z)
    R[:, 1, 2] = 2 * (y*z - r*x)
    R[:, 2, 0] = 2 * (x*z - r*y)
    R[:, 2, 1] = 2 * (y*z + r*x)
    R[:, 2, 2] = 1 - 2 * (x*x + y*y)
    return R


def build_scaling_rotation(s, r):
    """
    构建 3D 协方差矩阵的分解部分 L = R * S
    Sigma = L * L^T = R * S * S^T * R^T
    """
    L = torch.zeros((s.shape[0], 3, 3), device=s.device)
    R = build_rotation(r)

    L[:,0,0] = s[:,0]
    L[:,1,1] = s[:,1]
    L[:,2,2] = s[:,2]

    L = R @ L
    return L


def project_gaussians(means3D, scales, quats, viewmatrix, projmatrix, H, W, tan_fovx, tan_fovy):
    """
    将 3D 高斯投影到 2D (Batch)
    这是整个渲染器中最核心、计算量最大的部分
    """
    # 1. 转换到相机坐标系
    # means3D: (N, 3)
    # viewmatrix: (4, 4) World -> Camera

    # 齐次坐标 (N, 4)
    p_hom = torch.cat((means3D, torch.ones_like(means3D[:, :1])), dim=1)
    # (N, 4) @ (4, 4) -> (N, 4)
    p_cam = (p_hom @ viewmatrix.transpose(0, 1))[:, :3] 

    # 2. 剔除视锥体外的高斯 (简单的 Z 剔除)
    # z < 0.2 的高斯不渲染
    # DEBUG: 暂时禁用剔除，或者放宽条件
    mask = torch.ones_like(p_cam[:, 2], dtype=torch.bool) # p_cam[:, 2] > -999.0

    # 3. 构建 3D 协方差
    # L = R * S
    L = build_scaling_rotation(scales, quats)
    # Sigma = L @ L.T
    cov3D = L @ L.transpose(1, 2)

    # 4. 投影 3D 协方差到 2D (EWA Splatting)
    # J 矩阵 (雅可比)
    # J = [ fx/z   0    -(fx*x)/(z*z) ]
    #     [ 0    fy/z   -(fy*y)/(z*z) ]

    x = p_cam[:, 0]
    y = p_cam[:, 1]
    z = p_cam[:, 2]

    # DEBUG: 强制深度为正，防止投影反转
    z = torch.abs(z) + 1e-6

    # 焦距 f = W / (2 * tan(fov/2))
    fx = W / (2 * tan_fovx)
    fy = H / (2 * tan_fovy)

    # 构建 J 矩阵 (N, 2, 3)
    J = torch.zeros((means3D.shape[0], 2, 3), device=means3D.device)
    J[:, 0, 0] = fx / z
    J[:, 0, 2] = -(fx * x) / (z * z)
    J[:, 1, 1] = fy / z
    J[:, 1, 2] = -(fy * y) / (z * z)

    # 视图变换的旋转部分 W (3, 3)
    W_mat = viewmatrix[:3, :3] # (3, 3)

    # T = J @ W (N, 2, 3)
    # 注意：W 是所有高斯共用的，需要广播
    T = J @ W_mat.unsqueeze(0)

    # cov2D = T @ cov3D @ T.T
    cov2D = T @ cov3D @ T.transpose(1, 2)

    # 加上低通滤波 (防止协方差奇异)
    cov2D[:, 0, 0] += 0.3
    cov2D[:, 1, 1] += 0.3

    # 5. 计算屏幕坐标
    # u = fx * x / z + cx
    # v = fy * y / z + cy
    means2D = torch.zeros_like(means3D[:, :2])
    means2D[:, 0] = (x * fx / z) + W / 2
    means2D[:, 1] = (y * fy / z) + H / 2

    return means2D, cov2D, mask, p_cam[:, 2]

## 2. 鼠标怎样带着相机绕点云转？

`CameraController` 保存 yaw、pitch 和 distance：左右拖动改变水平角，上下拖动改变俯仰，滚轮改变距离。`get_view_matrix` 把这组状态转换成每一帧使用的视图矩阵。

模型同时保存点的位置、颜色、尺度、旋转和 opacity。下一节顺着渲染函数看，这些值最终怎样进入画布。

In [ ]:
class InteractiveGaussianModel(nn.Module):
    def __init__(self, num_gaussians):
        super().__init__()
        self.num_gaussians = num_gaussians

        # 初始化参数 (全部在 GPU 上)
        # DEBUG: 扩大分布范围到 [-5, 5]，让它们铺满屏幕
        self.means3D = nn.Parameter((torch.rand(num_gaussians, 3, device=device) - 0.5) * 10.0)

        # DEBUG: 颜色随机
        self.colors = nn.Parameter(torch.rand(num_gaussians, 3, device=device))

        # DEBUG: 缩放初始值设大一点 (1.0)，确保能看见
        self.scales = nn.Parameter(torch.ones(num_gaussians, 3, device=device) * 1.0)

        # 旋转: 随机四元数
        quats = torch.randn(num_gaussians, 4, device=device)
        self.quats = nn.Parameter(quats / quats.norm(dim=1, keepdim=True))

        # DEBUG: 不透明度设高一点 (sigmoid(2.0) ≈ 0.88)
        self.opacity_logit = nn.Parameter(torch.ones(num_gaussians, device=device) * 2.0)

    def forward(self):
        return self.means3D, torch.sigmoid(self.colors), torch.abs(self.scales), \
               self.quats / self.quats.norm(dim=1, keepdim=True), torch.sigmoid(self.opacity_logit)


class CameraController:
    def __init__(self):
        self.yaw = 0.0
        self.pitch = 0.0
        self.distance = 10.0
        self.last_mouse_pos = None
        self.is_dragging = False

    def handle_mouse(self, event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            self.is_dragging = True
            self.last_mouse_pos = (x, y)
        elif event == cv2.EVENT_LBUTTONUP:
            self.is_dragging = False
        elif event == cv2.EVENT_MOUSEMOVE:
            if self.is_dragging and self.last_mouse_pos is not None:
                dx = x - self.last_mouse_pos[0]
                dy = y - self.last_mouse_pos[1]
                self.yaw -= dx * 0.01
                self.pitch -= dy * 0.01
                self.last_mouse_pos = (x, y)
        elif event == cv2.EVENT_MOUSEWHEEL:
            # 注意：OpenCV 的滚轮事件处理可能因系统而异
            # 这里简单处理，如果 flags > 0 放大，< 0 缩小
            if flags > 0:
                self.distance *= 0.9
            else:
                self.distance *= 1.1

    def get_view_matrix(self):
        # 计算相机位置
        cam_x = self.distance * math.sin(self.yaw) * math.cos(self.pitch)
        cam_y = self.distance * math.sin(self.pitch)
        cam_z = self.distance * math.cos(self.yaw) * math.cos(self.pitch)

        eye = np.array([cam_x, cam_y, cam_z])
        target = np.array([0.0, 0.0, 0.0])
        up = np.array([0.0, 1.0, 0.0]) # Y-up

        # 构建 LookAt 矩阵
        z_axis = eye - target
        z_axis = z_axis / np.linalg.norm(z_axis)

        x_axis = np.cross(up, z_axis)
        x_axis = x_axis / np.linalg.norm(x_axis)

        y_axis = np.cross(z_axis, x_axis)

        view_mat = np.eye(4)
        view_mat[:3, 0] = x_axis
        view_mat[:3, 1] = y_axis
        view_mat[:3, 2] = z_axis
        view_mat[:3, 3] = -np.dot(np.array([x_axis, y_axis, z_axis]), eye)

        return torch.tensor(view_mat, dtype=torch.float32, device=device)

## 3. 放大看：这里画的是一个个 3×3 色块

渲染函数把投影坐标转成整数像素索引，再把颜色累加到中心附近的 3×3 小块。`index_put_(accumulate=True)` 负责把落在同一像素上的贡献叠起来。

这解释了窗口里的点为什么呈小方块，也解释了它怎样学习：颜色直接参与像素值计算，所以能收到梯度。

<details>
<summary>其余参数为什么留在原处？</summary>

坐标经过 `.long()` 后成为离散索引，位置到像素的梯度路径被切断；算出的 cov2D 和 opacity 没有参与着色。因而这个实验能拟合颜色，三维位置、尺度、旋转和 opacity 则不会通过当前像素损失学到变化。

连续高斯渲染会在周围像素上计算随位置、形状变化的权重。第二、三篇分别展示了足迹求值与可微优化，可用来对照这里的绘图方式。

</details>

In [ ]:
def render(model, view_matrix, H, W, fov_y):
    means3D, colors, scales, quats, opacities = model()

    tan_fovx = math.tan(fov_y * 0.5 * W / H)
    tan_fovy = math.tan(fov_y * 0.5)

    # 1. 投影 (Batch)
    means2D, cov2D, mask, depths = project_gaussians(
        means3D, scales, quats, view_matrix, None, H, W, tan_fovx, tan_fovy
    )

    # DEBUG: 打印投影信息 (只打印一次)
    if torch.rand(1) < 0.01: # 1% 的概率打印，避免刷屏
        print(f"\n--- Debug Info ---")
        print(f"Means3D range: X[{means3D[:,0].min():.2f}, {means3D[:,0].max():.2f}], Z[{means3D[:,2].min():.2f}, {means3D[:,2].max():.2f}]")
        print(f"Depths range: [{depths.min():.2f}, {depths.max():.2f}]")
        print(f"Means2D range: U[{means2D[:,0].min():.2f}, {means2D[:,0].max():.2f}], V[{means2D[:,1].min():.2f}, {means2D[:,1].max():.2f}]")
        print(f"Valid points: {mask.sum()} / {mask.numel()}")

    # 2. 简单的光栅化 (Additive Blending for speed)
    # 注意：这里为了实时性，我们仍然使用加法混合
    # 真正的排序+Alpha Blending 在 Python 里很难做到实时

    # 过滤掉视锥体外的高斯
    valid_means2D = means2D[mask]
    valid_cov2D = cov2D[mask]
    valid_colors = colors[mask]
    valid_opacities = opacities[mask]

    # 创建画布
    canvas = torch.zeros((H, W, 3), device=device)

    # 并行光栅化 (简化版)
    # 我们只画每个高斯中心的一个点，或者用很小的核
    # 为了演示效果，我们这里只画点云

    u = valid_means2D[:, 0].long()
    v = valid_means2D[:, 1].long()

    # 边界检查
    valid_indices = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    u = u[valid_indices]
    v = v[valid_indices]
    c = valid_colors[valid_indices]

    # 简单的 Scatter Add
    # canvas[v, u] += c # 这种写法在有重复索引时是不确定的
    # 正确做法是用 scatter_add，但为了代码简单，我们先这样写
    # 或者我们只显示前 10000 个点

    # DEBUG: 打印有多少个点在屏幕内
    # if u.numel() > 0:
    #     print(f"Points in screen: {u.numel()}, Max U: {u.max()}, Max V: {v.max()}")

    # 为了能看到东西，我们把点画大一点 (3x3)
    for i in range(-1, 2):
        for j in range(-1, 2):
            uu = torch.clamp(u + i, 0, W-1)
            vv = torch.clamp(v + j, 0, H-1)
            # 注意：这里用 accumulate=False，避免过曝
            # 或者我们除以一个系数
            canvas.index_put_((vv, uu), c * 0.5, accumulate=True)

    return torch.clamp(canvas, 0.0, 1.0)

## 4. 进入窗口：先看拟合，再转一转

开启第一格的 `RUN_INTERACTIVE` 后，下一格读取 `IP.png` 并进入事件循环。拟合模式使用固定的正面相机更新颜色；按空格切到查看模式，再用鼠标观察这组点。

可以留意：正面接近目标的颜色安排，换一个视角后会变成什么样？按 q 退出，代码会关闭窗口。

In [ ]:
if RUN_INTERACTIVE:
    img = Image.open("IP.png").convert("RGB").resize((320, 240))
    gt_img = torch.tensor(np.asarray(img).copy() / 255.0, dtype=torch.float32, device=device)
    H, W = gt_img.shape[:2]

    # 2. 初始化
    # 先用较少点观察交互行为
    model = InteractiveGaussianModel(NUM_POINTS)
    optimizer = optim.Adam(model.parameters(), lr=0.1)
    cam_ctrl = CameraController()

    # 3. OpenCV 窗口
    cv2.namedWindow("Interactive Training")
    cv2.setMouseCallback("Interactive Training", cam_ctrl.handle_mouse)

    print("Start Interactive Training...")
    print("Controls:")
    print("  [Space]: Toggle Training/View Mode")
    print("  [Mouse Left]: Rotate Camera")
    print("  [Mouse Wheel]: Zoom")
    print("  [q]: Quit")

    step = 0
    is_training = True

    # 固定一个训练视角 (正视图)
    training_cam = CameraController()
    training_cam.distance = 5.0 # 拉近相机，让物体看起来更大
    training_cam.yaw = 0.0
    training_cam.pitch = 0.0
    training_view_mat = training_cam.get_view_matrix()

    try:
        while True:
            if is_training:
                optimizer.zero_grad()

                # 训练模式：始终使用固定的正视图
                rendered = render(model, training_view_mat, H, W, math.pi/3)

                loss = torch.mean((rendered - gt_img)**2)
                loss.backward()
                optimizer.step()

                current_loss = loss.item()
                status_text = f"Training... Loss: {current_loss:.4f}"
            else:
                # 查看模式：使用鼠标控制的视角，不更新参数
                with torch.no_grad():
                    view_mat = cam_ctrl.get_view_matrix()
                    rendered = render(model, view_mat, H, W, math.pi/3)
                status_text = "View Mode (Not Training)"

            # 显示
            if step % 5 == 0: # 稍微提高刷新率
                img_np = (rendered.detach().cpu().numpy() * 255).astype(np.uint8)
                img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

                # 在图像上绘制状态文字
                cv2.putText(img_np, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.imshow("Interactive Training", img_np)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord(' '):
                is_training = not is_training
                print(f"\nSwitched to {'Training' if is_training else 'View'} Mode")

            step += 1
            if is_training:
                print(f"Step {step}, Loss: {current_loss:.4f}", end='\r')
            else:
                print(f"Step {step}, View Mode", end='\r')
    finally:
        cv2.destroyAllWindows()
else:
    print("Interactive demo is disabled; no window opened.")

## 从画点继续走向画高斯

现在已经能沿着“空间点 → 投影位置 → 像素索引 → 颜色”读懂交互画面。

下一步把 3×3 色块换成连续高斯权重，让协方差决定足迹、opacity 决定混合，再接入一致的相机与多视角数据，就能把更多几何参数纳入优化。

返回[相机与高斯投影](01_相机与高斯投影.ipynb)，把几何、着色和优化这三段重新连起来。